In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [2]:
!pip install -q tensorflow scikit-learn

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e10/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e10/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [4]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.replace("**", "", regex=False)
        .str.replace("&#x53;", "S", regex=False)
        .str.replace("&#x4C;", "L", regex=False)
        .str.replace("&#x43;", "C", regex=False)
        .str.strip()
    )
    return df

train = clean_columns(train)
test = clean_columns(test)

target = "satisfaction"
id_col = "id"

y = (
    train[target].astype(str).str.lower()
    .map({"true": 1, "false": 0, "1": 1, "0": 0})
    .astype("float32")
    .values
)

X = train.drop(columns=[target, id_col]).copy()
X_test = test.drop(columns=[id_col]).copy()

cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

num_cols = [c for c in X.columns if c not in cat_cols]

for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")
    med = X[col].median()
    X[col] = X[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

encoders = {}
cat_sizes = []

for col in cat_cols:
    enc = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    combined = pd.concat(
        [X[[col]], X_test[[col]]],
        axis=0
    )
    enc.fit(combined)

    X[col] = enc.transform(X[[col]]).astype("int32") + 1
    X_test[col] = enc.transform(X_test[[col]]).astype("int32") + 1

    encoders[col] = enc
    cat_sizes.append(int(combined[col].nunique()) + 2)

scaler = StandardScaler()

X_num = scaler.fit_transform(X[num_cols]).astype("float32")
X_test_num = scaler.transform(X_test[num_cols]).astype("float32")

X_cat = X[cat_cols].values.astype("int32")
X_test_cat = X_test[cat_cols].values.astype("int32")

X_train_num, X_val_num, X_train_cat, X_val_cat, y_train, y_val = train_test_split(
    X_num,
    X_cat,
    y,
    test_size=0.15,
    random_state=SEED,
    stratify=y
)

d_model = 64
num_features = len(num_cols)
num_categories = len(cat_cols)


In [5]:
def transformer_block(x, heads=4, ff_dim=128, dropout=0.15):
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=heads,
        key_dim=d_model // heads,
        dropout=dropout
    )(x, x)

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(
        x + tf.keras.layers.Dropout(dropout)(attn)
    )

    ff = tf.keras.layers.Dense(
        ff_dim,
        activation="gelu"
    )(x)

    ff = tf.keras.layers.Dense(d_model)(ff)

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(
        x + tf.keras.layers.Dropout(dropout)(ff)
    )

    return x

num_input = tf.keras.Input(
    shape=(num_features,),
    name="numeric"
)

cat_input = tf.keras.Input(
    shape=(num_categories,),
    dtype="int32",
    name="categorical"
)

num_tokens = tf.keras.layers.Reshape(
    (num_features, 1)
)(num_input)

num_tokens = tf.keras.layers.Dense(d_model)(num_tokens)

cat_tokens = []

for i, size in enumerate(cat_sizes):
    inp = tf.keras.layers.Lambda(
        lambda z, idx=i: z[:, idx:idx + 1]
    )(cat_input)

    token = tf.keras.layers.Embedding(
        input_dim=size,
        output_dim=d_model
    )(inp)

    cat_tokens.append(token)

if cat_tokens:
    cat_tokens = tf.keras.layers.Concatenate(axis=1)(cat_tokens)
    x = tf.keras.layers.Concatenate(axis=1)(
        [num_tokens, cat_tokens]
    )
else:
    x = num_tokens

x = tf.keras.layers.LayerNormalization()(x)

for _ in range(3):
    x = transformer_block(x)

x = tf.keras.layers.GlobalAveragePooling1D()(x)

x = tf.keras.layers.Dense(
    128,
    activation="gelu"
)(x)

x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Dense(
    64,
    activation="gelu"
)(x)

x = tf.keras.layers.Dropout(0.2)(x)

output = tf.keras.layers.Dense(
    1,
    activation="sigmoid"
)(x)

model = tf.keras.Model(
    inputs=[num_input, cat_input],
    outputs=output
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=0.0005,
        weight_decay=0.0001
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(
            curve="ROC",
            name="roc_auc"
        )
    ]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc",
        mode="max",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_roc_auc",
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    [X_train_num, X_train_cat],
    y_train,
    validation_data=(
        [X_val_num, X_val_cat],
        y_val
    ),
    epochs=100,
    batch_size=512,
    callbacks=callbacks,
    verbose=1
)

val_pred = model.predict(
    [X_val_num, X_val_cat],
    batch_size=2048,
    verbose=0
).ravel()

print("Validation ROC-AUC:", roc_auc_score(y_val, val_pred))

I0000 00:00:1791533776.864227      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791533776.868251      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/100
   7/1162 ━━━━━━━━━━━━━━━━━━━━ 24s 21ms/step - loss: 0.6152 - roc_auc: 0.7314

I0000 00:00:1791533803.646316      86 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1162/1162 ━━━━━━━━━━━━━━━━━━━━ 59s 30ms/step - loss: 0.3653 - roc_auc: 0.9113 - val_loss: 0.3466 - val_roc_auc: 0.9206 - learning_rate: 5.0000e-04
Epoch 2/100
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - loss: 0.3370 - roc_auc: 0.9243 - val_loss: 0.3255 - val_roc_auc: 0.9336 - learning_rate: 5.0000e-04
Epoch 3/100
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - loss: 0.2992 - roc_auc: 0.9377 - val_loss: 0.2851 - val_roc_auc: 0.9429 - learning_rate: 5.0000e-04
Epoch 4/100
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - loss: 0.2864 - roc_auc: 0.9422 - val_loss: 0.2764 - val_roc_auc: 0.9467 - learning_rate: 5.0000e-04
Epoch 5/100
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - loss: 0.2762 - roc_auc: 0.9455 - val_loss: 0.2697 - val_roc_auc: 0.9482 - learning_rate: 5.0000e-04
Epoch 6/100
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - loss: 0.2698 - roc_auc: 0.9469 - val_loss: 0.2652 - val_roc_auc: 0.9487 - learning_rate: 5.0000e-04
Epoch 7/100
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step -

In [6]:

test_pred = model.predict(
    [X_test_num, X_test_cat],
    batch_size=2048,
    verbose=0
).ravel()

submission = pd.DataFrame({
    "id": test[id_col],
    "satisfaction": np.clip(test_pred, 1e-6, 1 - 1e-6)
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(submission.head())
print("Submission shape:", submission.shape)
print("Saved: /kaggle/working/submission.csv")

       id  satisfaction
0  699635      0.376450
1  699636      0.961486
2  699637      0.931381
3  699638      0.043854
4  699639      0.045905
Submission shape: (299844, 2)
Saved: /kaggle/working/submission.csv
